In [1]:
import cv2 as cv
import time

In [ ]:
GRID_SIZE = 5

# Motion detection parameters
MOTION_THRESHOLD = 45
MIN_ACTIVE_PIXELS = 100

# Background subtractor settings
HISTORY = 500
VAR_THRESHOLD = 25

# Reset after 10 seconds of complete inactivity
RESET_INTERVAL = 10.0

# Camera
CAMERA_INDEX = 0

In [ ]:
cap = cv.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    print("Could not open camera.")
    exit()

In [ ]:
bg_sub = cv.createBackgroundSubtractorMOG2(
    history=HISTORY,
    varThreshold=VAR_THRESHOLD,
    detectShadows=True
)

In [ ]:
# Contains the cells that have already become active.
active_cells = set()

# Time when significant motion was last detected anywhere
last_motion_time = time.time()

# Used for calculating FPS
prev_time = time.time()
fps = 0

In [ ]:
while True:

    ret, frame = cap.read()

    if not ret:
        print("Could not read frame.")
        break

    height, width = frame.shape[:2]

    cell_width = width // GRID_SIZE
    cell_height = height // GRID_SIZE

    motion_mask = bg_sub.apply(frame)

    _, motion_mask = cv.threshold(
        motion_mask,
        200,
        255,
        cv.THRESH_BINARY
    )

    # Clean small noise using morphology
    kernel = cv.getStructuringElement(
        cv.MORPH_ELLIPSE,
        (5, 5)
    )

    motion_mask = cv.morphologyEx(
        motion_mask,
        cv.MORPH_OPEN,
        kernel
    )

    motion_mask = cv.morphologyEx(
        motion_mask,
        cv.MORPH_CLOSE,
        kernel
    )

    # DETECT MOTION IN EACH CELL
    motion_cells = set()

    for row in range(GRID_SIZE):

        for col in range(GRID_SIZE):

            # Calculate cell boundaries

            x1 = col * cell_width
            y1 = row * cell_height

            # Make sure the last cell reaches the image edge

            if col == GRID_SIZE - 1:
                x2 = width
            else:
                x2 = (col + 1) * cell_width

            if row == GRID_SIZE - 1:
                y2 = height
            else:
                y2 = (row + 1) * cell_height


            # Extract this cell from the motion mask

            cell_mask = motion_mask[y1:y2, x1:x2]


            # Count the number of detected motion pixels
            active_pixel_count = cv.countNonZero(cell_mask)


            # Significant motion?
            if active_pixel_count >= MIN_ACTIVE_PIXELS:

                cell_index = row * GRID_SIZE + col

                motion_cells.add(cell_index)

    # CHECK FOR ANY MOTION ANYWHERE
    if len(motion_cells) > 0:

        # There is significant motion somewhere in the frame.

        last_motion_time = time.time()

    # CONTAGION RULE
    if len(active_cells) == 0:

        # NO SEED YET
        #
        # The first cell with significant motion becomes
        # the seed.

        if len(motion_cells) > 0:

            # Pick one of the detected motion cells.
            #
            # sorted() makes the result deterministic for
            # demonstration purposes.

            seed = sorted(motion_cells)[0]

            active_cells.add(seed)


    else:
        # A SEED ALREADY EXISTS
        #
        # Only neighbouring cells can become active.

        eligible_cells = set()

        for active_cell in active_cells:

            # Convert cell number back into row/column

            active_row = active_cell // GRID_SIZE
            active_col = active_cell % GRID_SIZE


            # Check all 8 neighbouring positions

            for row_offset in [-1, 0, 1]:

                for col_offset in [-1, 0, 1]:

                    # Skip the cell itself

                    if row_offset == 0 and col_offset == 0:
                        continue


                    neighbour_row = active_row + row_offset
                    neighbour_col = active_col + col_offset


                    # Make sure neighbour is inside the grid

                    if (
                        0 <= neighbour_row < GRID_SIZE
                        and
                        0 <= neighbour_col < GRID_SIZE
                    ):

                        neighbour_index = (
                            neighbour_row * GRID_SIZE
                            + neighbour_col
                        )

                        eligible_cells.add(neighbour_index)

        # Activate only eligible cells where motion occurred

        new_cells = motion_cells.intersection(eligible_cells)

        active_cells.update(new_cells)

    # RESET AFTER 10 SECONDS OF COMPLETE INACTIVITY

    inactive_time = time.time() - last_motion_time

    if inactive_time >= RESET_INTERVAL:

        active_cells.clear()

        # Restart the timer

        last_motion_time = time.time()

    # CREATE THE OUTPUT IMAGE

    output_frame = frame.copy()

    # Invert active cells

    for cell_index in active_cells:

        row = cell_index // GRID_SIZE
        col = cell_index % GRID_SIZE

        x1 = col * cell_width
        y1 = row * cell_height

        if col == GRID_SIZE - 1:
            x2 = width
        else:
            x2 = (col + 1) * cell_width

        if row == GRID_SIZE - 1:
            y2 = height
        else:
            y2 = (row + 1) * cell_height


        # Extract the cell

        cell = output_frame[y1:y2, x1:x2]

        # Invert its colours

        output_frame[y1:y2, x1:x2] = cv.bitwise_not(cell)

    # DRAW GRID

    for i in range(1, GRID_SIZE):

        x = i * cell_width

        cv.line(
            output_frame,
            (x, 0),
            (x, height),
            (255, 255, 255),
            1
        )

    for i in range(1, GRID_SIZE):

        y = i * cell_height

        cv.line(
            output_frame,
            (0, y),
            (width, y),
            (255, 255, 255),
            1
        )

    # CALCULATE FPS

    current_time = time.time()

    elapsed = current_time - prev_time

    if elapsed > 0:

        fps = 1 / elapsed

    previous_time = current_time

    # DETERMINE ELIGIBLE CELLS FOR DISPLAY

    eligible_cells = set()

    if len(active_cells) > 0:

        for active_cell in active_cells:

            active_row = active_cell // GRID_SIZE
            active_col = active_cell % GRID_SIZE

            for row_offset in [-1, 0, 1]:

                for col_offset in [-1, 0, 1]:

                    if row_offset == 0 and col_offset == 0:
                        continue

                    neighbour_row = active_row + row_offset
                    neighbour_col = active_col + col_offset

                    if (
                        0 <= neighbour_row < GRID_SIZE
                        and
                        0 <= neighbour_col < GRID_SIZE
                    ):

                        neighbour_index = (
                            neighbour_row * GRID_SIZE
                            + neighbour_col
                        )

                        eligible_cells.add(neighbour_index)

        # Don't label already active cells as merely eligible

        eligible_cells -= active_cells

    # DRAW CELL STATE INFORMATION

    for cell_index in eligible_cells:

        row = cell_index // GRID_SIZE
        col = cell_index % GRID_SIZE

        x1 = col * cell_width
        y1 = row * cell_height

        # Draw an "E" for eligible

        cv.putText(
            output_frame,
            "E",
            (x1 + 8, y1 + 25),
            cv.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 255),
            2
        )

    # DISPLAY INFORMATION

    reset_remaining = max(
        0,
        RESET_INTERVAL - inactive_time
    )

    cv.putText(
        output_frame,
        f"FPS: {fps:.1f}",
        (10, 25),
        cv.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    cv.putText(
        output_frame,
        f"Active cells: {len(active_cells)}",
        (10, 55),
        cv.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    cv.putText(
        output_frame,
        f"Eligible cells: {len(eligible_cells)}",
        (10, 85),
        cv.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv.putText(
        output_frame,
        f"Reset in: {reset_remaining:.1f}s",
        (10, 115),
        cv.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    # SHOW THE RESULT

    cv.imshow("Motion Contagion Grid", output_frame)

    if cv.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv.destroyAllWindows()